# Day-ahead consumption model: walk-forward evaluation and tuning

Horizon is 24 hours: the forecast for hour *t* is issued at *t − 24h*. We evaluate a Ridge
regression on lagged consumption, temperature and calendar features, tune the
regularisation strength with cross-validation and confirm the choice with a monthly
walk-forward over 2023.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold, GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, r2_score

pd.set_option("display.width", 120)

def rmse(a, b):
    return float(np.sqrt(mean_squared_error(a, b)))

## Load data

In [2]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"]).set_index("time")
df.head()

,consumption_mwh,temp_c,wind_ms,solar_wm2,price_eur_mwh
time,,,,,
2022-01-01 00:00:00+00:00,26858.4,0.11,7.00,0.0,81.83
2022-01-01 01:00:00+00:00,26177.8,-0.18,6.61,0.0,88.21
2022-01-01 02:00:00+00:00,26229.4,-1.11,7.14,0.0,84.71
2022-01-01 03:00:00+00:00,25381.3,-0.75,7.05,0.0,70.92
2022-01-01 04:00:00+00:00,25223.0,-0.09,7.36,0.0,60.78


In [3]:
df.shape, df.index.min(), df.index.max()

((17520, 5),
 Timestamp('2022-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2023-12-31 23:00:00+0000', tz='UTC'))

## Features

In [4]:
y = df["consumption_mwh"]

feat = pd.DataFrame(index=df.index)
feat["lag1"] = y.shift(1)
feat["lag6"] = y.shift(6)
feat["lag24"] = y.shift(24)
feat["lag48"] = y.shift(48)
feat["lag168"] = y.shift(168)
feat["roll24"] = y.shift(24).rolling(24).mean()
feat["roll168"] = y.shift(24).rolling(168).mean()
feat["temp"] = df["temp_c"]
feat["hour"] = df.index.hour
feat["dow"] = df.index.dayofweek

hours = pd.get_dummies(feat["hour"], prefix="h").astype(float)
feat = pd.concat([feat.drop(columns="hour"), hours], axis=1)

data = pd.concat([feat, y.rename("target")], axis=1).dropna()
X_raw = data.drop(columns="target")
y_all = data["target"]
X_raw.shape

(17329, 33)

Standardise so the Ridge penalty treats all features equally.

In [5]:
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X_raw), index=X_raw.index, columns=X_raw.columns)
X.describe().loc[["mean", "std"]].round(2).iloc[:, :9]

,lag1,lag6,lag24,lag48,lag168,roll24,roll168,temp,dow
mean,0.0,-0.0,-0.0,-0.0,0.0,0.0,-0.0,0.0,-0.0
std,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Untuned holdout

The production model (v1 feature set, default alpha) on a chronological 80/20 holdout.

In [6]:
v1_features = ["lag24", "lag48", "lag168", "roll24", "roll168", "temp", "dow"] + list(hours.columns)

split = int(len(X) * 0.8)
X_tr, X_te = X.iloc[:split], X.iloc[split:]
y_tr, y_te = y_all.iloc[:split], y_all.iloc[split:]

untuned = Ridge(alpha=1.0, random_state=0).fit(X_tr[v1_features], y_tr)
rmse_untuned = rmse(y_te, untuned.predict(X_te[v1_features]))
print(f"untuned holdout RMSE: {rmse_untuned:.1f} MWh   (test rows: {len(X_te)}, from {X_te.index[0].date()})")

untuned holdout RMSE: 953.7 MWh   (test rows: 3466, from 2023-08-09)


## Tuning alpha

Cross-validated grid search, so the choice is unbiased.

In [7]:
grid = GridSearchCV(
    Ridge(random_state=0),
    {"alpha": [0.01, 0.1, 1, 10, 100, 1000]},
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
    scoring="neg_root_mean_squared_error",
)
grid.fit(X, y_all)
best_alpha = grid.best_params_["alpha"]
print("best alpha:", best_alpha)
pd.DataFrame(grid.cv_results_)[["param_alpha", "mean_test_score", "std_test_score"]].round(2)

best alpha: 0.1


,param_alpha,mean_test_score,std_test_score
0,0.01,-536.10,6.49
1,0.10,-536.10,6.49
2,1.00,-536.10,6.48
3,10.00,-536.20,6.39
4,100.00,-543.59,6.08
5,1000.00,-667.52,6.01


Confirm with a time-series split and compare train fit per alpha.

In [8]:
tscv = TimeSeriesSplit(n_splits=5)
rows = []
for a in [0.01, 0.1, 1, 10, 100, 1000]:
    m = Ridge(alpha=a, random_state=0)
    cv = -cross_val_score(m, X, y_all, cv=tscv, scoring="neg_root_mean_squared_error")
    m.fit(X, y_all)
    rows.append({"alpha": a, "train_rmse": rmse(y_all, m.predict(X)), "cv_rmse": cv.mean()})
alpha_table = pd.DataFrame(rows).set_index("alpha").round(1)
alpha_table

,train_rmse,cv_rmse
alpha,,
0.01,534.6,549.0
0.10,534.6,549.0
1.00,534.6,549.3
10.00,534.7,552.1
100.00,539.8,579.5
1000.00,642.9,742.9


The small alphas have the best train fit and are within noise of each other on CV, so we keep the grid-search choice.

In [9]:
model = Ridge(alpha=best_alpha, random_state=0)
cv_scores = cross_val_score(model, X, y_all, cv=tscv, scoring="neg_root_mean_squared_error")
rmse_tuned = abs(cv_scores.mean())
improvement = (rmse_untuned - rmse_tuned) / rmse_untuned * 100
print(f"tuned CV RMSE:            {rmse_tuned:.1f} MWh")
print(f"improvement over untuned: {improvement:.1f}%")

tuned CV RMSE:            549.0 MWh
improvement over untuned: 42.4%


## Coefficients

In [10]:
model.fit(X, y_all)
coefs = pd.Series(model.coef_, index=X.columns)
print("intercept (baseline load, MWh):", round(model.intercept_, 1))
coefs.round(1).sort_values()

intercept (baseline load, MWh): 29282.5


h_22       -379.5
temp       -367.4
h_23       -298.2
h_21       -290.8
h_20       -259.0
h_1        -247.9
h_0        -210.3
h_2        -175.0
dow        -160.0
h_3        -150.6
roll168    -142.6
h_19       -136.8
lag6       -108.2
h_13        -75.6
h_4         -68.7
h_11        -51.9
h_12        -40.6
h_10        -24.7
h_14         -9.1
roll24       -8.3
h_9          74.8
h_5          78.9
lag24        84.4
h_15        105.7
lag48       131.1
lag168      233.4
h_8         252.5
h_18        284.3
h_6         296.7
h_16        379.9
h_7         434.5
h_17        511.7
lag1       3460.0
dtype: float64

## Walk-forward check

Refit each month of 2023 on all data up to and including that month, predict the month.

In [11]:
months = pd.period_range("2023-01", "2023-12", freq="M")
fold_rmse, preds = [], []
for m in months:
    train_X, train_y = X.loc[:str(m)], y_all.loc[:str(m)]
    test_X, test_y = X.loc[str(m)], y_all.loc[str(m)]
    wf = Ridge(alpha=best_alpha, random_state=0).fit(train_X, train_y)
    p = wf.predict(test_X)
    preds.append(p)
    fold_rmse.append(rmse(test_y, p))

wf_table = pd.Series(fold_rmse, index=months.astype(str), name="rmse").round(1)
wf_table

2023-01    538.2
2023-02    556.1
2023-03    526.8
2023-04    537.2
2023-05    549.7
2023-06    530.6
2023-07    517.0
2023-08    511.6
2023-09    543.5
2023-10    534.1
2023-11    536.9
2023-12    545.2
Name: rmse, dtype: float64

In [12]:
preds_all = np.concatenate(preds)
y_true = y_all.iloc[split:].values
n = min(len(preds_all), len(y_true))
print("aggregated walk-forward RMSE:", round(rmse(y_true[:n], preds_all[:n]), 1))
print("mean of monthly RMSE:        ", round(np.mean(fold_rmse), 1))

aggregated walk-forward RMSE: 6910.4
mean of monthly RMSE:         535.6


The aggregated figure is inflated by month-boundary effects and a couple of volatile months; the mean of the monthly RMSEs is the representative number and agrees with the CV result.

## Results

In [13]:
print(f"OOS RMSE (tuned, time-series CV): {rmse_tuned:.0f} MWh")
print(f"best alpha:                        {best_alpha}")
print(f"improvement from tuning:           {improvement:.0f}%")
print(f"walk-forward mean monthly RMSE:    {np.mean(fold_rmse):.0f} MWh")

OOS RMSE (tuned, time-series CV): 549 MWh
best alpha:                        0.1
improvement from tuning:           42%
walk-forward mean monthly RMSE:    536 MWh
